# Khởi tạo dữ liệu (Setup Data)

In [1]:
import sqlite3
import pandas as pd

# 1. Tạo kết nối cơ sở dữ liệu SQLite trong bộ nhớ
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# 2. Tạo bảng course và student
cursor.execute('''
CREATE TABLE course (
    id INTEGER PRIMARY KEY,
    course_name TEXT
)
''')

cursor.execute('''
CREATE TABLE student (
    student_id INTEGER PRIMARY KEY,
    name TEXT,
    class TEXT,
    course_id INTEGER,
    score REAL
)
''')

# 3. Chèn dữ liệu vào bảng course
course_data = [
    (12, 'Giai tich'),
    (34, 'Thong ke'),
    (26, 'Tin hoc')
]
cursor.executemany('INSERT INTO course VALUES (?, ?)', course_data)

# 4. Chèn dữ liệu vào bảng student (Sử dụng None cho các ô trống)
student_data = [
    (1, 'Nguyen Minh Hoang', 'May Tinh', 12, 6.7),
    (2, 'Tran Thi Lan', 'Kinh Te', 34, 9.2),
    (3, 'Pham Van Nam', 'Toan Tin', None, 7.9),
    (4, 'Le Thanh Huyen', 'Toan Tin', 20, 7.2),
    (5, 'Vu Quoc Anh', 'May Tinh', 24, 8.0),
    (6, 'Dang Thuy Linh', 'May Tinh', 24, 5.5),
    (7, 'Bui Tien Dung', 'Kinh Te', 34, 9.2),
    (8, 'Ho Ngoc Mai', 'Toan Tin', 20, 8.8),
    (9, 'Duong Huu Phuc', 'Kinh Te', None, 7.2),
    (10, 'Cao Thi Hanh', 'May Tinh', None, 7.0)
]
cursor.executemany('INSERT INTO student VALUES (?, ?, ?, ?, ?)', student_data)
conn.commit()

print("Đã khởi tạo dữ liệu thành công!")

Đã khởi tạo dữ liệu thành công!


# Yêu Cầu 1: Kết nối hai bảng

1.1. Tích Descartes (Cross Join)

In [2]:
query_descartes = '''
SELECT * FROM student
CROSS JOIN course;
'''
display(pd.read_sql_query(query_descartes, conn))

,student_id,name,class,course_id,score,id,course_name
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,12,Giai tich
1,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,26,Tin hoc
2,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,34,Thong ke
3,2,Tran Thi Lan,Kinh Te,34.0,9.2,12,Giai tich
4,2,Tran Thi Lan,Kinh Te,34.0,9.2,26,Tin hoc
5,2,Tran Thi Lan,Kinh Te,34.0,9.2,34,Thong ke
6,3,Pham Van Nam,Toan Tin,NaN,7.9,12,Giai tich
7,3,Pham Van Nam,Toan Tin,NaN,7.9,26,Tin hoc
8,3,Pham Van Nam,Toan Tin,NaN,7.9,34,Thong ke
9,4,Le Thanh Huyen,Toan Tin,20.0,7.2,12,Giai tich


1.2. Các loại JOIN

In [4]:
# INNER JOIN (Giữ nguyên)
query_inner = '''SELECT * FROM student INNER JOIN course ON student.course_id = course.id;'''
print("--- INNER JOIN ---")
display(pd.read_sql_query(query_inner, conn))

# LEFT JOIN (Giữ nguyên)
query_left = '''SELECT * FROM student LEFT JOIN course ON student.course_id = course.id;'''
print("--- LEFT JOIN ---")
display(pd.read_sql_query(query_left, conn))

# RIGHT JOIN (Giả lập bằng cách đảo bảng: LEFT JOIN ngược)
# Thay vì Student RIGHT JOIN Course, ta dùng Course LEFT JOIN Student
query_right_emu = '''
SELECT student.*, course.* FROM course
LEFT JOIN student ON student.course_id = course.id;
'''
print("--- RIGHT JOIN (Emulated) ---")
display(pd.read_sql_query(query_right_emu, conn))

# FULL OUTER JOIN (Giả lập bằng cách: LEFT JOIN UNION ALL với các bản ghi chỉ có ở bảng phải)
query_full_emu = '''
SELECT * FROM student LEFT JOIN course ON student.course_id = course.id
UNION ALL
SELECT * FROM course LEFT JOIN student ON student.course_id = course.id
WHERE student.student_id IS NULL;
'''
print("--- FULL OUTER JOIN (Emulated) ---")
display(pd.read_sql_query(query_full_emu, conn))

--- INNER JOIN ---


,student_id,name,class,course_id,score,id,course_name
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,12,Giai tich
1,2,Tran Thi Lan,Kinh Te,34,9.2,34,Thong ke
2,7,Bui Tien Dung,Kinh Te,34,9.2,34,Thong ke


--- LEFT JOIN ---


,student_id,name,class,course_id,score,id,course_name
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,12.0,Giai tich
1,2,Tran Thi Lan,Kinh Te,34.0,9.2,34.0,Thong ke
2,3,Pham Van Nam,Toan Tin,NaN,7.9,NaN,None
3,4,Le Thanh Huyen,Toan Tin,20.0,7.2,NaN,None
4,5,Vu Quoc Anh,May Tinh,24.0,8.0,NaN,None
5,6,Dang Thuy Linh,May Tinh,24.0,5.5,NaN,None
6,7,Bui Tien Dung,Kinh Te,34.0,9.2,34.0,Thong ke
7,8,Ho Ngoc Mai,Toan Tin,20.0,8.8,NaN,None
8,9,Duong Huu Phuc,Kinh Te,NaN,7.2,NaN,None
9,10,Cao Thi Hanh,May Tinh,NaN,7.0,NaN,None


--- RIGHT JOIN (Emulated) ---


,student_id,name,class,course_id,score,id,course_name
0,1.0,Nguyen Minh Hoang,May Tinh,12.0,6.7,12,Giai tich
1,NaN,None,None,NaN,NaN,26,Tin hoc
2,7.0,Bui Tien Dung,Kinh Te,34.0,9.2,34,Thong ke
3,2.0,Tran Thi Lan,Kinh Te,34.0,9.2,34,Thong ke


--- FULL OUTER JOIN (Emulated) ---


,student_id,name,class,course_id,score,id,course_name
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,12.0,Giai tich
1,2,Tran Thi Lan,Kinh Te,34.0,9.2,34.0,Thong ke
2,3,Pham Van Nam,Toan Tin,NaN,7.9,NaN,None
3,4,Le Thanh Huyen,Toan Tin,20.0,7.2,NaN,None
4,5,Vu Quoc Anh,May Tinh,24.0,8.0,NaN,None
5,6,Dang Thuy Linh,May Tinh,24.0,5.5,NaN,None
6,7,Bui Tien Dung,Kinh Te,34.0,9.2,34.0,Thong ke
7,8,Ho Ngoc Mai,Toan Tin,20.0,8.8,NaN,None
8,9,Duong Huu Phuc,Kinh Te,NaN,7.2,NaN,None
9,10,Cao Thi Hanh,May Tinh,NaN,7.0,NaN,None


# Yêu Cầu 2: Cập nhật, Xóa và Thống kê

Bước 2.1: Cập nhật và Xóa bản ghi

In [5]:
# Cập nhật giá trị thiếu bằng mã hợp lệ có trong bảng course (VD: mã 26)
cursor.execute('''
UPDATE student
SET course_id = 26
WHERE course_id IS NULL;
''')

# Loại bỏ bản ghi có course_id không tồn tại trong bảng course
cursor.execute('''
DELETE FROM student
WHERE course_id NOT IN (SELECT id FROM course);
''')
conn.commit()
print("Đã cập nhật dữ liệu và xóa các bản ghi không hợp lệ.")

Đã cập nhật dữ liệu và xóa các bản ghi không hợp lệ.


Bước 2.2: Các truy vấn thống kê (a, b, c)

In [6]:
# a. Tổng số sinh viên, điểm trung bình của từng lớp
query_2a = '''
SELECT class, COUNT(student_id) AS tong_sinh_vien, ROUND(AVG(score), 2) AS diem_tb_lop
FROM student
GROUP BY class;
'''
print("2a. Thống kê theo lớp:")
display(pd.read_sql_query(query_2a, conn))

# b. Tổng số sinh viên, điểm trung bình của từng môn học
query_2b = '''
SELECT c.course_name, COUNT(s.student_id) AS tong_sinh_vien, ROUND(AVG(s.score), 2) AS diem_tb_mon
FROM student s
JOIN course c ON s.course_id = c.id
GROUP BY c.course_name;
'''
print("2b. Thống kê theo môn học:")
display(pd.read_sql_query(query_2b, conn))

# c. Phân loại thi đua theo điểm của từng môn học
query_2c = '''
SELECT c.course_name, s.name, s.score,
    CASE
        WHEN s.score >= 9.0 THEN 'Xuất sắc'
        WHEN s.score >= 6.0 AND s.score <= 8.9 THEN 'Tốt'
        ELSE 'Kém'
    END AS phan_loai
FROM student s
JOIN course c ON s.course_id = c.id;
'''
print("2c. Phân loại sinh viên:")
display(pd.read_sql_query(query_2c, conn))

2a. Thống kê theo lớp:


,class,tong_sinh_vien,diem_tb_lop
0,Kinh Te,3,8.53
1,May Tinh,2,6.85
2,Toan Tin,1,7.90


2b. Thống kê theo môn học:


,course_name,tong_sinh_vien,diem_tb_mon
0,Giai tich,1,6.70
1,Thong ke,2,9.20
2,Tin hoc,3,7.37


2c. Phân loại sinh viên:


,course_name,name,score,phan_loai
0,Giai tich,Nguyen Minh Hoang,6.7,Tốt
1,Thong ke,Tran Thi Lan,9.2,Xuất sắc
2,Tin hoc,Pham Van Nam,7.9,Tốt
3,Thong ke,Bui Tien Dung,9.2,Xuất sắc
4,Tin hoc,Duong Huu Phuc,7.2,Tốt
5,Tin hoc,Cao Thi Hanh,7.0,Tốt


# Yêu Cầu 3: Xếp hạng sinh viên

In [7]:
# a. Xếp hạng theo Điểm số tổng (và lấy Top 3 cao nhất / thấp nhất)
query_3a = '''
SELECT student_id, name, score, RANK() OVER (ORDER BY score DESC) as rank_toan_truong
FROM student;
'''
df_3a = pd.read_sql_query(query_3a, conn)
print("Xếp hạng toàn trường:")
display(df_3a)
print("Top 3 Cao nhất:"); display(df_3a.head(3))
print("Top 3 Thấp nhất:"); display(df_3a.tail(3))

# b. Xếp hạng theo Lớp
query_3b = '''
SELECT class, name, score, RANK() OVER (PARTITION BY class ORDER BY score DESC) as rank_trong_lop
FROM student;
'''
print("Xếp hạng theo từng lớp:")
display(pd.read_sql_query(query_3b, conn))
# (Để lọc top 3 từng lớp, bạn có thể bọc câu SQL trên làm Subquery và lấy điều kiện WHERE rank_trong_lop <= 3)

# c. Xếp hạng theo Môn học
query_3c = '''
SELECT c.course_name, s.name, s.score, RANK() OVER (PARTITION BY s.course_id ORDER BY s.score DESC) as rank_trong_mon
FROM student s
JOIN course c ON s.course_id = c.id;
'''
print("Xếp hạng theo từng môn học:")
display(pd.read_sql_query(query_3c, conn))

Xếp hạng toàn trường:


,student_id,name,score,rank_toan_truong
0,2,Tran Thi Lan,9.2,1
1,7,Bui Tien Dung,9.2,1
2,3,Pham Van Nam,7.9,3
3,9,Duong Huu Phuc,7.2,4
4,10,Cao Thi Hanh,7.0,5
5,1,Nguyen Minh Hoang,6.7,6


Top 3 Cao nhất:


,student_id,name,score,rank_toan_truong
0,2,Tran Thi Lan,9.2,1
1,7,Bui Tien Dung,9.2,1
2,3,Pham Van Nam,7.9,3


Top 3 Thấp nhất:


,student_id,name,score,rank_toan_truong
3,9,Duong Huu Phuc,7.2,4
4,10,Cao Thi Hanh,7.0,5
5,1,Nguyen Minh Hoang,6.7,6


Xếp hạng theo từng lớp:


,class,name,score,rank_trong_lop
0,Kinh Te,Tran Thi Lan,9.2,1
1,Kinh Te,Bui Tien Dung,9.2,1
2,Kinh Te,Duong Huu Phuc,7.2,3
3,May Tinh,Cao Thi Hanh,7.0,1
4,May Tinh,Nguyen Minh Hoang,6.7,2
5,Toan Tin,Pham Van Nam,7.9,1


Xếp hạng theo từng môn học:


,course_name,name,score,rank_trong_mon
0,Giai tich,Nguyen Minh Hoang,6.7,1
1,Tin hoc,Pham Van Nam,7.9,1
2,Tin hoc,Duong Huu Phuc,7.2,2
3,Tin hoc,Cao Thi Hanh,7.0,3
4,Thong ke,Tran Thi Lan,9.2,1
5,Thong ke,Bui Tien Dung,9.2,1


# Yêu Cầu 4: Bổ sung trường graduation_date

In [9]:
# Thêm cột graduation_date
cursor.execute('''
ALTER TABLE student ADD COLUMN graduation_date DATETIME;
''')

# Cập nhật graduation_date = Thời gian hiện tại + (Thứ hạng theo điểm) ngày
cursor.execute('''
WITH RankedStudents AS (
    SELECT student_id, RANK() OVER (ORDER BY score DESC) as ranking
    FROM student
)
UPDATE student
SET graduation_date = (
    SELECT datetime('now', '+' || ranking || ' days')
    FROM RankedStudents
    WHERE RankedStudents.student_id = student.student_id
);
''')
conn.commit()

# Hiển thị bảng student sau khi cập nhật
query_4 = 'SELECT student_id, name, score, graduation_date FROM student;'
print("Kết quả bảng student sau khi cập nhật thời gian tốt nghiệp:")
display(pd.read_sql_query(query_4, conn))

Kết quả bảng student sau khi cập nhật thời gian tốt nghiệp:


,student_id,name,score,graduation_date
0,1,Nguyen Minh Hoang,6.7,2026-04-07 17:23:44
1,2,Tran Thi Lan,9.2,2026-04-02 17:23:44
2,3,Pham Van Nam,7.9,2026-04-04 17:23:44
3,7,Bui Tien Dung,9.2,2026-04-02 17:23:44
4,9,Duong Huu Phuc,7.2,2026-04-05 17:23:44
5,10,Cao Thi Hanh,7.0,2026-04-06 17:23:44
